# Portfolio Allocation Methods - Complete Comparison

**Runtime:** ~12 minutes  
**Level:** Intermediate

This notebook provides a comprehensive comparison of all portfolio allocation algorithms in RustyBT.

## Available Allocation Methods

RustyBT provides five allocation algorithms:

1. **FixedAllocation** - Static weights (e.g., 60/40)
2. **DynamicAllocation** - Weights based on recent performance metrics
3. **RiskParityAllocation** - Equal risk contribution from each strategy
4. **KellyCriterionAllocation** - Optimal growth portfolio
5. **DrawdownBasedAllocation** - Reduce allocation to strategies in drawdown

## When to Use Each Method

| Method | Best For | Pros | Cons |
|--------|----------|------|------|
| **Fixed** | Stable strategies, simple setup | Predictable, easy | Doesn't adapt |
| **Dynamic** | Varying market conditions | Adaptive | May chase performance |
| **Risk Parity** | Balanced risk exposure | Diversification | Requires volatility estimation |
| **Kelly** | Growth maximization | Theoretically optimal | Aggressive, assumes accuracy |
| **Drawdown-Based** | Risk management focus | Protects capital | May miss recoveries |

---

**📋 Notebook Information**

- **RustyBT Version:** 0.1.2+
- **Last Validated:** 2025-11-07
- **API Compatibility:** Verified ✅
- **Documentation:** [Portfolio API Reference](https://rustybt.readthedocs.io/en/latest/api/)

---

In [ ]:
# Setup
from rustybt.analytics import setup_notebook
setup_notebook()

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

from rustybt import TradingAlgorithm
from rustybt.portfolio import (
    PortfolioAllocator,
    FixedAllocation,
    DynamicAllocation,
    RiskParityAllocation,
    KellyCriterionAllocation,
    DrawdownBasedAllocation,
    AllocationRebalancer,
    RebalancingFrequency,
)

print("✓ Imports successful")

## 1. Define Sample Sub-Strategies

Create three simple strategies with different characteristics.

In [ ]:
class MomentumStrategy(TradingAlgorithm):
    """High volatility, trend-following strategy."""
    
    def initialize(self, context):
        context.asset = self.symbol('QQQ')
        self.schedule_function(
            self.rebalance,
            date_rules=self.date_rules.month_start(),
            time_rules=self.time_rules.market_open()
        )
    
    def rebalance(self, context, data):
        if not data.can_trade(context.asset):
            return
        
        prices = data.history(context.asset, 'close', 60, '1d')
        if len(prices) < 60:
            return
        
        # Momentum signal
        returns = (prices[-1] - prices[0]) / prices[0]
        
        if returns > 0:
            self.order_target_percent(context.asset, 1.0)
        else:
            self.order_target_percent(context.asset, 0.0)


class MeanReversionStrategy(TradingAlgorithm):
    """Low volatility, mean reversion strategy."""
    
    def initialize(self, context):
        context.asset = self.symbol('SPY')
        self.schedule_function(
            self.rebalance,
            date_rules=self.date_rules.week_start(),
            time_rules=self.time_rules.market_open()
        )
    
    def rebalance(self, context, data):
        if not data.can_trade(context.asset):
            return
        
        prices = data.history(context.asset, 'close', 20, '1d')
        if len(prices) < 20:
            return
        
        # Mean reversion signal
        mean = prices.mean()
        current = prices[-1]
        
        if current < mean * 0.98:  # Buy when below mean
            self.order_target_percent(context.asset, 1.0)
        elif current > mean * 1.02:  # Sell when above mean
            self.order_target_percent(context.asset, 0.0)


class DefensiveStrategy(TradingAlgorithm):
    """Ultra-low volatility, defensive strategy."""
    
    def initialize(self, context):
        context.asset = self.symbol('TLT')  # Treasury bonds
        # Simply hold
        self.order_target_percent(context.asset, 1.0)


print("✓ Sub-strategies defined")

## 2. Fixed Allocation

Static weights that never change.

In [ ]:
# Define fixed weights
fixed_weights = {
    'momentum': 0.40,
    'mean_reversion': 0.40,
    'defensive': 0.20,
}

# Create allocator
fixed_allocation = FixedAllocation(weights=fixed_weights)

print("Fixed Allocation:")
print(f"  Momentum:       {fixed_weights['momentum']:.1%}")
print(f"  Mean Reversion: {fixed_weights['mean_reversion']:.1%}")
print(f"  Defensive:      {fixed_weights['defensive']:.1%}")
print(f"\nTotal: {sum(fixed_weights.values()):.1%}")

**When to use Fixed Allocation:**
- Simple, predictable allocation
- Strategies have stable characteristics
- You want to maintain specific risk exposure
- Regulatory or policy requirements for fixed weights

## 3. Dynamic Allocation

Adjust weights based on recent performance metrics (Sharpe, Sortino, etc.).

In [ ]:
# Create dynamic allocator
dynamic_allocation = DynamicAllocation(
    lookback=60,  # Use last 60 days of performance
    metric='sharpe',  # Allocate based on Sharpe ratio
    min_weight=0.05,  # Minimum 5% to each strategy
    max_weight=0.70,  # Maximum 70% to any strategy
)

print("Dynamic Allocation Configuration:")
print(f"  Lookback period: 60 days")
print(f"  Metric: Sharpe Ratio")
print(f"  Weight range: [5%, 70%]")
print(f"\nAllocation Logic:")
print("  1. Calculate each strategy's Sharpe over lookback")
print("  2. Convert to positive scores (handle negative Sharpe)")
print("  3. Allocate proportionally to scores")
print("  4. Apply min/max constraints")

**When to use Dynamic Allocation:**
- Market regimes change frequently
- Some strategies perform better in certain conditions
- You want to "tilt" toward better-performing strategies
- You can tolerate some momentum-chasing

## 4. Risk Parity Allocation

Each strategy contributes equal risk to the portfolio.

In [ ]:
# Create risk parity allocator
risk_parity_allocation = RiskParityAllocation(
    lookback=90,  # Use last 90 days for volatility estimation
    min_weight=0.05,
    max_weight=0.80,
)

print("Risk Parity Allocation Configuration:")
print(f"  Lookback period: 90 days")
print(f"  Weight range: [5%, 80%]")
print(f"\nAllocation Logic:")
print("  1. Calculate each strategy's volatility")
print("  2. Set weights inversely proportional to volatility")
print("  3. Result: Each strategy contributes equal risk")
print(f"\nExample:")
print("  If Strategy A has 2x volatility of Strategy B,")
print("  Strategy A gets 0.5x weight of Strategy B")

**When to use Risk Parity:**
- You want balanced risk contribution
- Strategies have very different volatilities
- Classic diversification approach
- Institutional-grade allocation

## 5. Kelly Criterion Allocation

Maximize long-term growth rate (theoretically optimal).

In [ ]:
# Create Kelly allocator
kelly_allocation = KellyCriterionAllocation(
    lookback=120,  # Longer lookback for Kelly estimation
    kelly_fraction=0.5,  # Use half-Kelly for safety
    min_weight=0.0,  # Can reduce to 0%
    max_weight=0.60,  # Cap at 60% (Kelly can be aggressive)
)

print("Kelly Criterion Allocation Configuration:")
print(f"  Lookback period: 120 days")
print(f"  Kelly fraction: 0.5 (Half-Kelly)")
print(f"  Weight range: [0%, 60%]")
print(f"\nAllocation Logic:")
print("  1. Calculate expected return and variance for each strategy")
print("  2. Apply Kelly formula: f = (μ - r) / σ²")
print("  3. Scale by kelly_fraction (0.5 = half-Kelly)")
print("  4. Apply constraints")
print(f"\nNote: Half-Kelly recommended for practical use")
print("  Full Kelly can be very aggressive")

**When to use Kelly Criterion:**
- Growth maximization is priority
- You trust your return/volatility estimates
- Long-term compounding focus
- Use fractional Kelly (e.g., 0.5x) for safety

## 6. Drawdown-Based Allocation

Reduce allocation to strategies experiencing drawdowns.

In [ ]:
# Create drawdown-based allocator
drawdown_allocation = DrawdownBasedAllocation(
    lookback=30,  # Monitor last 30 days for drawdowns
    base_weights={'momentum': 0.4, 'mean_reversion': 0.4, 'defensive': 0.2},
    drawdown_threshold=0.05,  # Reduce allocation if drawdown > 5%
    reduction_factor=0.5,  # Cut allocation in half when in drawdown
)

print("Drawdown-Based Allocation Configuration:")
print(f"  Lookback period: 30 days")
print(f"  Base weights: Momentum 40%, MR 40%, Defensive 20%")
print(f"  Drawdown threshold: 5%")
print(f"  Reduction factor: 0.5 (cut in half)")
print(f"\nAllocation Logic:")
print("  1. Start with base weights")
print("  2. If strategy drawdown > threshold:")
print("     - Multiply weight by reduction_factor")
print("  3. Redistribute freed capital to other strategies")
print(f"\nExample:")
print("  If Momentum has 10% drawdown (> 5% threshold):")
print("  - Momentum: 40% → 20%")
print("  - Extra 20% redistributed to MR and Defensive")

**When to use Drawdown-Based:**
- Risk management is primary concern
- Protect capital during strategy drawdowns
- Strategies can have extended losing periods
- Conservative approach

## 7. Comparing All Methods

Run backtests with each allocation method and compare results.

In [ ]:
def run_portfolio_comparison(strategies, allocations, start_date, end_date, bundle):
    """
    Compare all allocation methods side-by-side.
    """
    results = {}
    
    for name, allocation in allocations.items():
        print(f"Running {name}...")
        
        # Create portfolio with this allocation
        portfolio = PortfolioAllocator(
            strategies=strategies,
            allocation=allocation,
        )
        
        # Run backtest
        # result = run_algorithm(
        #     start=start_date,
        #     end=end_date,
        #     capital_base=100000.0,
        #     bundle=bundle,
        #     algorithm=portfolio,
        # )
        
        # results[name] = result
    
    return results

# Define allocation methods to compare
allocations = {
    'Fixed': fixed_allocation,
    'Dynamic (Sharpe)': dynamic_allocation,
    'Risk Parity': risk_parity_allocation,
    'Kelly (Half)': kelly_allocation,
    'Drawdown-Based': drawdown_allocation,
}

# Run comparison
# comparison_results = run_portfolio_comparison(
#     strategies={
#         'momentum': MomentumStrategy(),
#         'mean_reversion': MeanReversionStrategy(),
#         'defensive': DefensiveStrategy(),
#     },
#     allocations=allocations,
#     start_date=pd.Timestamp('2020-01-01', tz='utc'),
#     end_date=pd.Timestamp('2023-12-31', tz='utc'),
#     bundle='yfinance',
# )

print("✓ Comparison framework defined")

### Visualize Comparison Results

In [ ]:
def plot_allocation_comparison(results):
    """
    Create comprehensive comparison visualizations.
    """
    # Extract metrics
    metrics = pd.DataFrame({
        name: {
            'Total Return': result['total_return'].iloc[-1],
            'CAGR': result['cagr'].iloc[-1],
            'Sharpe': result['sharpe_ratio'].iloc[-1],
            'Sortino': result['sortino_ratio'].iloc[-1],
            'Calmar': result['calmar_ratio'].iloc[-1],
            'Max Drawdown': result['max_drawdown'].min(),
            'Volatility': result['volatility'].iloc[-1],
        }
        for name, result in results.items()
    }).T
    
    # Create subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Equity Curves',
            'Risk-Adjusted Returns',
            'Return vs Volatility',
            'Max Drawdown Comparison'
        ),
        specs=[
            [{'type': 'scatter'}, {'type': 'bar'}],
            [{'type': 'scatter'}, {'type': 'bar'}]
        ]
    )
    
    # 1. Equity curves
    for name, result in results.items():
        fig.add_trace(
            go.Scatter(
                x=result.index,
                y=result['portfolio_value'],
                name=name,
                mode='lines'
            ),
            row=1, col=1
        )
    
    # 2. Sharpe ratios
    fig.add_trace(
        go.Bar(
            x=metrics.index,
            y=metrics['Sharpe'],
            name='Sharpe',
            marker_color='blue'
        ),
        row=1, col=2
    )
    
    # 3. Return vs Volatility scatter
    fig.add_trace(
        go.Scatter(
            x=metrics['Volatility'],
            y=metrics['CAGR'],
            mode='markers+text',
            text=metrics.index,
            textposition='top center',
            marker=dict(size=12),
            name='Strategies'
        ),
        row=2, col=1
    )
    
    # 4. Max drawdowns
    fig.add_trace(
        go.Bar(
            x=metrics.index,
            y=metrics['Max Drawdown'],
            name='Max DD',
            marker_color='red'
        ),
        row=2, col=2
    )
    
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Method", row=1, col=2)
    fig.update_xaxes(title_text="Volatility", row=2, col=1)
    fig.update_xaxes(title_text="Method", row=2, col=2)
    
    fig.update_yaxes(title_text="Portfolio Value", row=1, col=1)
    fig.update_yaxes(title_text="Sharpe Ratio", row=1, col=2)
    fig.update_yaxes(title_text="CAGR", row=2, col=1)
    fig.update_yaxes(title_text="Max Drawdown", row=2, col=2)
    
    fig.update_layout(
        height=900,
        title_text="Portfolio Allocation Methods Comparison",
        showlegend=True
    )
    
    # Print summary table
    print("\n=== Performance Summary ===")
    print(metrics.to_string())
    
    # Rank by Sharpe
    ranked = metrics.sort_values('Sharpe', ascending=False)
    print(f"\n=== Rankings (by Sharpe) ===")
    for i, (name, row) in enumerate(ranked.iterrows(), 1):
        print(f"{i}. {name}: {row['Sharpe']:.3f}")
    
    return fig, metrics

# Example usage:
# fig, metrics = plot_allocation_comparison(comparison_results)
# fig.show()

print("✓ Visualization function defined")

## 8. Allocation Weight Evolution

Visualize how weights change over time for dynamic methods.

In [ ]:
def plot_weight_evolution(portfolio_results, method_name):
    """
    Plot how allocation weights evolve over time.
    """
    # Extract weight history from portfolio
    weights_df = portfolio_results['weights_history']
    
    fig = go.Figure()
    
    # Plot each strategy's weight
    for strategy in weights_df.columns:
        fig.add_trace(go.Scatter(
            x=weights_df.index,
            y=weights_df[strategy],
            name=strategy,
            mode='lines',
            stackgroup='one',  # Create stacked area chart
        ))
    
    fig.update_layout(
        title=f"Allocation Weight Evolution: {method_name}",
        xaxis_title="Date",
        yaxis_title="Weight",
        yaxis=dict(tickformat='.0%'),
        height=500,
        hovermode='x unified'
    )
    
    # Print statistics
    print(f"\n=== Weight Statistics: {method_name} ===")
    print("\nMean weights:")
    print(weights_df.mean().to_string())
    print("\nWeight volatility (std):")
    print(weights_df.std().to_string())
    print("\nWeight range:")
    print(f"Min: {weights_df.min().to_string()}")
    print(f"Max: {weights_df.max().to_string()}")
    
    return fig

# Example usage:
# fig = plot_weight_evolution(dynamic_results, 'Dynamic (Sharpe)')
# fig.show()

print("✓ Weight evolution function defined")

## Summary

### Method Selection Guide

**Choose Fixed Allocation when:**
- ✅ You want predictable, easy-to-understand allocation
- ✅ Strategies have stable characteristics
- ✅ Regulatory requirements for fixed weights

**Choose Dynamic Allocation when:**
- ✅ Market regimes change frequently
- ✅ Strategies perform differently in different conditions
- ✅ You want adaptive portfolio management
- ⚠️ Be careful of performance-chasing

**Choose Risk Parity when:**
- ✅ Strategies have very different volatilities
- ✅ You want balanced risk contribution
- ✅ Institutional-grade diversification approach
- ✅ Works well for most portfolios

**Choose Kelly Criterion when:**
- ✅ Growth maximization is the goal
- ✅ You trust your return/volatility estimates
- ⚠️ Use fractional Kelly (0.25-0.5) for safety
- ⚠️ Can be aggressive

**Choose Drawdown-Based when:**
- ✅ Capital preservation is priority
- ✅ You want to reduce exposure during strategy drawdowns
- ✅ Conservative risk management approach
- ⚠️ May miss early recovery periods

### Typical Performance Characteristics

Based on empirical testing:

**Sharpe Ratio (typically):**
1. Risk Parity - Often highest (balanced risk)
2. Kelly (fractional) - High but volatile
3. Dynamic - Good in trending markets
4. Fixed - Stable baseline
5. Drawdown-Based - Lower but consistent

**Max Drawdown (typically):**
1. Drawdown-Based - Lowest (by design)
2. Risk Parity - Low (balanced)
3. Fixed - Moderate
4. Dynamic - Can be high (performance-chasing)
5. Kelly - Can be highest (aggressive)

### Best Practices

1. **Start with Risk Parity** - Good default for most portfolios
2. **Compare methods** - Backtest all methods before choosing
3. **Use constraints** - Always set min/max weights
4. **Monitor weights** - Track weight evolution over time
5. **Combine approaches** - Hybrid methods can work well
6. **Rebalance regularly** - But not too frequently (monthly/quarterly)
7. **Consider transaction costs** - Factor in rebalancing costs

### Next Steps

- Notebook 13: Portfolio Optimization + Walk Forward
- Notebook 08: Portfolio Construction Basics
- Documentation: [Portfolio Allocation Guide](https://rustybt.readthedocs.io/en/latest/portfolio/allocation/)

### Resources

- [Portfolio Allocation API](https://rustybt.readthedocs.io/en/latest/api/portfolio/)
- [Risk Parity Guide](https://rustybt.readthedocs.io/en/latest/guides/risk-parity/)
- [Kelly Criterion Guide](https://rustybt.readthedocs.io/en/latest/guides/kelly/)